## This script removes fully duplicated rows from merged_raw_data_outliers_removed.csv and then aggregates rows with the same station_ID and observation_date by computing the mean:

In [18]:
# load required modules:

import pandas as pd
from glob import glob
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import warnings
from tqdm import trange
from time import time
import pickle
#from scipy.spatial.distance import cdist
#from scipy.spatial import distance_matrix
import pdb
from math import sin, cos, sqrt, atan2, radians
#from scipy.spatial.distance import pdist, squareform
#from sklearn.metrics.pairwise import pairwise_distances
import multiprocessing
from multiprocessing import Pool
n_cores = multiprocessing.cpu_count()
import os
from tqdm import tqdm
from time import time
import os

In [19]:
# define the output folder path:
output_folder = '../../output_data/merged_datasets'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

### read in merged_raw_data_outliers_removed.csv 

In [2]:
#--------------- WATER QUALITY DATA------------ ########################
# read in merged raw data:
tic = time()

raw_data = pd.read_csv('../../output_data/merged_datasets/merged_raw_data_outliers_removed.csv')
raw_data = raw_data.set_index(['dataset', 'site_id']).replace(-9999, np.nan).drop('X', axis = 1, errors = 'ignore').sort_index()
#raw_data['obs_date'] = pd.DatetimeIndex(raw_data['obs_date'].values)
print('Loaded raw data in [sec]: ', f'{time() -tic}')



# ----convert site id's to string----

raw_data = raw_data.reset_index()
raw_data['site_id'] = raw_data['site_id'].astype(str)
raw_data = raw_data.set_index(['dataset', 'site_id'])


# ---change the Ob' of arcticdeltas data to Ob ---
raw_data = raw_data.reset_index()
raw_data['site_id'] = raw_data['site_id'].replace('Ob\'','Ob').values

# get sorted index
raw_data = raw_data.set_index(['dataset', 'site_id']).sort_index()
raw_data_index = raw_data.sort_index().index.unique()
print("Block time [sec]: ",f'{time()-tic}')


C:\Users\bartusch\AppData\Local\Temp\ipykernel_19792\1290736427.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv('../../output_data/merged_datasets/merged_raw_data_outliers_removed.csv')


Loaded raw data in [sec]:  9.673640727996826
Block time [sec]:  13.656761884689331


### replace negative values and zero values by nan: 

In [3]:
columns_fractions = ['NO3N', 'NO2N', 'NO2N_NO3N', 'NH4N', 'DIN', 'TOC', 'DOC', 'TP', 'DIP', 'OPO4']

# Iterate over each column in the list
for column in columns_fractions:
    raw_data.loc[(raw_data[column] <= 0), column] = np.nan


### Drop rows where all columns of interest (columns with observed parameters) have NA values:

In [6]:
columns_fractions = ['NO3N', 'NO2N', 'NO2N_NO3N', 'NH4N', 'DIN', 'TOC', 'DOC', 'TP', 'DIP', 'OPO4']
# Now drop all columns which have in all columns of interest only nans after removing outliers:
columns_subset = [col for col in columns_fractions if col in raw_data.columns]
# Drop rows where all columns in the subset are NaNs
raw_data.dropna(subset=columns_subset, how='all', inplace=True)

In [7]:
len(raw_data)

3384761

#### Check raw_data dataframe for duplicated rows and merge rows if they have the same station_id and same observation date 'obs_date': 

In [11]:
# delete unnamed columns:
raw_data2 = raw_data.drop(['Unnamed: 0.1', 'Unnamed: 0'], axis = 1, errors = 'ignore')
# set station_ID as index:
raw_data2['station_ID'] = raw_data2.index
raw_data2['station_ID'] = raw_data2['station_ID'].astype(str)


In [9]:
raw_data2[raw_data2['station_ID']=="('GRQA', 'BEL00050')"]


obs_date  NO3N   NH4N  NO2N  TOC  DOC     TP  DIP  year  \
dataset site_id                                                               
GRQA    BEL00050  2005-01-12   NaN  0.425   NaN  NaN  NaN  0.370  NaN  2005   
        BEL00050  2005-02-09   NaN  0.425   NaN  NaN  NaN  0.495  NaN  2005   
        BEL00050  2005-03-14   NaN  1.080   NaN  NaN  NaN  0.230  NaN  2005   
        BEL00050  2005-04-12   NaN  0.140   NaN  NaN  NaN  0.235  NaN  2005   
        BEL00050  2005-05-09   NaN  0.140   NaN  NaN  NaN  0.260  NaN  2005   
...                      ...   ...    ...   ...  ...  ...    ...  ...   ...   
        BEL00050  2012-08-06   NaN  0.080   NaN  NaN  5.0  0.280  NaN  2012   
        BEL00050  2012-09-10   NaN  0.080   NaN  NaN  6.8  0.270  NaN  2012   
        BEL00050  2012-10-22   NaN  0.080   NaN  NaN  6.0  0.630  NaN  2012   
        BEL00050  2012-11-12   NaN  0.080   NaN  NaN  5.2  0.280  NaN  2012   
        BEL00050  2012-12-10   NaN  0.254   NaN  NaN  6.0  0.230  NaN  2012   

                  NO3  ...  NO2N_F  TOC_F  DOC_F  TP_F  DIP_F  NO2N_NO3N  DIN  \
dataset site_id        ...                                                      
GRQA    BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
...               ...  ...     ...    ...    ...   ...    ...        ...  ...   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   
        BEL00050  NaN  ...     NaN    NaN    NaN   NaN    NaN        NaN  NaN   

                   Q  OPO4            station_ID  
dataset site_id                                   
GRQA    BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
...               ..   ...                   ...  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  
        BEL00050 NaN   NaN  ('GRQA', 'BEL00050')  

[103 rows x 22 columns]

In [10]:
# reset index and remove completely duplicated rows from the dataframe:
raw_data3 = raw_data2.reset_index()
raw_data4 = raw_data3.drop_duplicates(keep = 'first')

print("\nIn raw data occured fully duplicated rows:\n")
print(f'{(len(raw_data3)-len(raw_data4))} rows were duplicated and removed.')


In raw data occured fully duplicated rows:

2391 rows were duplicated and removed.


### In the following steps process the DataFrame raw_data4, which has already been filtered for fully duplicated rows 
### If multiple rows share the same `observation_date`, `station_id`, but have differing observed values,
### they are grouped and aggregated by calculating the mean.
###
### If each compound is recorded in its own row (long format), these are also aggregated into a single row per group.


In [12]:
# get all duplicated rows (obs_date and station_ID):


# use raw_data4: and select rows that have same site_id, dataset and observation date: 
rows_dup =raw_data4.duplicated(subset=['station_ID', 'obs_date'], keep=False)
duplicated = raw_data4[rows_dup]


# reset index
duplicated = duplicated.reset_index()

# convert site_id to 
duplicated['site_id'] = duplicated['site_id'].astype(str)
# set non numeric columns --> to treat as non numeric during aggregation--> take first value
non_numeric_columns = ['dataset', 'site_id', 'year']

# set the numeric columns --> to treat as numeric during aggregation --> mean
numeric_columns = ['NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP', 'NO3N_F', 'NH4N_F', 'NO2N_F', 'TOC_F', 'DOC_F', 'TP_F', 'DIP_F', 'NO2N_NO3N', 'DIN', 'Q','OPO4']

# Define the aggregation methods for each column
aggregation = {column: 'mean' for column in numeric_columns}
aggregation.update({column: 'first' for column in non_numeric_columns})

#  group data by station_ID and obs_date and aggregate duplicated rows by calculationg mean:
aggregated_data = duplicated.groupby(['station_ID', 'obs_date'], as_index = True).agg(aggregation).reset_index()

# convert site_id to string type:   
aggregated_data['site_id'] = aggregated_data['site_id'].astype(str)


In [15]:
# print example of duplicated rows:
duplicated[0:10]
duplicated[(duplicated['station_ID']=="('GRQA', '102360')") & (duplicated['obs_date']=='1993-04-13')]

,index,dataset,site_id,obs_date,NO3N,NH4N,NO2N,TOC,DOC,TP,...,NO2N_F,TOC_F,DOC_F,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4,station_ID
0,94994,GRQA,102360,1993-04-13,NaN,0.020002,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"('GRQA', '102360')"
30,95024,GRQA,102360,1993-04-13,NaN,NaN,0.010001,NaN,2.800004,0.059995,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"('GRQA', '102360')"


## subset raw_data4 and keep only not duplicated rows:

In [16]:
# delete duplicated rows from raw_data4 and add new rows of dataframe aggregated_data:

# subset raw_data4 and keep only not duplicated rows: 
not_duplicated_raw_data = raw_data4[~rows_dup]
not_duplicated_raw_data = not_duplicated_raw_data.reset_index()

# set column order corresponding not duplicated_raw_data: 
col_order = not_duplicated_raw_data.columns.to_list()
aggregated_data = aggregated_data.reindex(columns=col_order)



In [17]:
# append aggregated data to not_duplicated_raw_data:
#filtered_raw_data = not_duplicated_raw_data.append(aggregated_data, ignore_index=True)
filtered_raw_data = pd.concat([not_duplicated_raw_data, aggregated_data], ignore_index=True)
print(len(filtered_raw_data))


3314094


#### Save filtered and aggregated data:

In [20]:
# set  columns as before and save data as filtered_merged_raw_data: 

# important convert column site_id to string type in order to avoid wrong read in: otherwise 0122399 will be read in as 122399
filtered_raw_data['site_id'] = filtered_raw_data['site_id'].astype(str)

# drop column 'station_ID' 
filtered_raw_data = filtered_raw_data.drop('station_ID', axis = 1)

# 
#filtered_raw_save = filtered_raw_data.copy()
#filtered_raw_save = filtered_raw_save.drop('year', axis = 1)
# save as csv-file: 
filtered_raw_data.to_csv(f'{output_folder}/filtered_merged_raw_data.csv', index = False)

In [19]:
filtered_raw_data.describe()

,index,NO3N,NH4N,NO2N,TOC,DOC,TP,DIP,year,NO3,...,NH4N_F,NO2N_F,TOC_F,DOC_F,TP_F,DIP_F,NO2N_NO3N,DIN,Q,OPO4
count,3.246135e+06,2.000032e+06,1.122485e+06,936933.000000,656500.000000,870390.000000,2.038907e+06,1.265642e+06,3.314094e+06,0.0,...,267110.000000,267110.00000,267110.000000,267110.000000,267110.000000,267110.000000,5396.000000,100.000000,282330.000000,138942.000000
mean,1.736497e+06,2.249406e+00,2.567494e-01,0.049253,8.375175,5.795838,3.494592e-01,1.218014e-01,1.995021e+03,NaN,...,0.019636,0.08176,0.000124,0.009704,0.011666,0.079319,0.025293,0.024530,272.548891,0.083104
std,9.622782e+05,1.093677e+01,1.399505e+00,0.299383,18.974065,10.529989,1.576198e+00,5.817909e-01,1.553615e+01,NaN,...,0.139554,0.27415,0.011769,0.123422,0.107654,0.280221,0.030221,0.035115,1687.736001,0.216726
min,0.000000e+00,1.000000e-04,8.000000e-06,0.000093,0.019998,0.002100,5.000000e-05,2.000000e-05,1.900000e+03,NaN,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000885,0.001000,0.000000,0.000500
25%,9.371055e+05,2.999319e-01,2.000200e-02,0.008000,3.300000,2.400000,4.000000e-02,1.601304e-02,1.985000e+03,NaN,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.010570,0.005000,0.821189,0.012000
50%,1.748639e+06,1.129433e+00,5.900000e-02,0.020000,5.500000,3.999999,1.000000e-01,3.913200e-02,1.997000e+03,NaN,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.017250,0.016000,9.882579,0.036000
75%,2.562112e+06,3.321018e+00,1.400000e-01,0.040000,9.000000,6.400000,2.200000e-01,9.000000e-02,2.006000e+03,NaN,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.029945,0.027250,62.297062,0.079000
max,3.384760e+06,1.040000e+04,1.602133e+02,50.000000,1534.000000,870.840000,4.850000e+02,4.040391e+02,2.022000e+03,NaN,...,2.000000,2.00000,2.000000,2.000000,2.000000,2.000000,0.787440,0.236000,63712.904330,6.800000


In [20]:
# print number of unique site_ids:
filtered_raw_data['site_id'].nunique()

73197

### Check data availability of merged and filtered data:

In [21]:

# convert df to long format: 
filtered_raw_data_2 = filtered_raw_data.copy()
filtered_raw_data_2 = filtered_raw_data_2.drop(columns = ['NO3N_F', 'NO2N_F', 'NH4N_F', 'DOC_F', 'TOC_F', 'DIP_F', 'TP_F', 'Q'], axis = 1)

df_long = pd.melt(filtered_raw_data_2, id_vars=['obs_date', 'year', 'site_id', 'dataset'], var_name='fraction', value_name='value').replace(-9999,np.nan) .dropna(subset=['value'])
#PER FRACTION:
    
    
# n observations , start year and end year per fraction:
stats_per_fraction = df_long.groupby('fraction')['year'].agg(['count', 'min', 'max'])
    
    
#n unique site ids per fraction:
    
n_stations_per_fraction = df_long.groupby('fraction')['site_id'].nunique()
    
    
    
# per fraction statistics: mean, median, range number of observations per station, 

mean_count_per_fraction_site = df_long.groupby(['fraction', 'site_id']).count().groupby(level=[0])['obs_date'].agg(['mean', 'median', 'max', 'min'])
mean_count_per_fraction_site
    
    

#mean and median time series length in years; median/mean start year; maximum time series length years per station;
    
stats_years_station_frac = df_long.groupby(['site_id', 'fraction'])['year'].nunique().groupby(level=[1]).agg(['mean', 'median', 'max', 'min']).rename(columns = {'mean':'mean_station_years',
                                                                                                                                                                 'median':'median_station_years',
                                                                                                                                                                 'max':'max_station_years',
                                                                                                                                                                 'min':'min_station_years'})
    
# median number of samples per station and year and fraction
median_obs_fraction_station_year = df_long.groupby(['site_id', 'fraction','year'])['value'].count().groupby(level = [1]).median()
merged_df1 = stats_years_station_frac.merge(median_obs_fraction_station_year, left_index=True, right_index=True).rename(columns = {'value':'median_n_obs_per_year'})
merged_df2 = merged_df1.merge(stats_per_fraction,  left_index=True, right_index=True).rename(columns = {'count':'n_obs','min':'start_year','max':'end_year'})
merged_df3 = merged_df2.merge(mean_count_per_fraction_site, left_index=True, right_index=True).rename(columns = {'mean':'mean_obs_station',
                                                                                                                     'median':'median_obs_station',
                                                                                                                     'max':'max_obs_station',
                                                                                                                     'min':'min_obs_station'})
stats_fractions = merged_df3.merge(n_stations_per_fraction, left_index=True, right_index=True).rename(columns = {'site_id':'n_sites'})

stats_fractions = stats_fractions.reset_index()
   


In [22]:
stats_fractions.T
#stats_fractions.T.to_csv('../../../stats_filtered_merged_raw_data_per_fraction_for_report.csv')

,0,1,2,3,4,5,6,7,8,9,10
fraction,DIN,DIP,DOC,NH4N,NO2N,NO2N_NO3N,NO3N,OPO4,TOC,TP,index
mean_station_years,2.0,6.538721,4.397021,5.525562,4.187159,9.222222,4.810184,22.719178,4.380896,4.845533,5.131337
median_station_years,2.0,3.0,2.0,2.0,2.0,9.0,2.0,22.0,2.0,2.0,2.0
max_station_years,2,52,43,52,55,15,69,41,50,56,77
min_station_years,2,1,1,1,1,3,1,10,1,1,1
median_n_obs_per_year,12.5,10.0,6.0,8.0,5.0,25.0,5.0,11.0,6.0,6.0,6.0
n_obs,100,1265642,870390,1122485,936933,5396,2000032,138942,656500,2038907,3246135
start_year,2008,1942,1958,1942,1900,2004,1900,1966,1905,1900,1900
end_year,2009,2022,2022,2022,2020,2021,2022,2013,2020,2022,2022
mean_obs_station,25.0,72.3348,35.039855,60.468944,26.878564,199.851852,37.248706,475.828767,29.861269,40.811606,44.854085


In [23]:
### END

## Check higher data availability after aggregating data:
## Compare `filtered_merged_raw_data` to `merged_raw_data` to evaluate the effect of aggregation on data availability.

## 1. check number of stations with observed DIN:DOC/TOC:SRP in unfiltered raw data:

In [26]:

raw_available = raw_data2[(raw_data2['NO3N'].notna()|raw_data2['NO2N_NO3N'].notna())&(raw_data2['NH4N'].notna())&(raw_data2['TOC'].notna()|raw_data2['DOC'].notna())&(raw_data2['DIP'].notna()|raw_data2['OPO4'].notna())]['station_ID'].unique()
print(f'In unfiltered raw data were {len(raw_available)} stations with observed DIN:DOC/TOC:SRP (OPO4/DIP) data available.')
#raw_data2[(raw_data2['NO3N'].notna()|raw_data2['NO2N_NO3N'].notna())&(raw_data2['NH4N'].notna())&(raw_data2['TOC'].notna()|raw_data2['DOC'].notna())&(raw_data2['DIP'].notna()|raw_data2['OPO4'].notna())]

In unfiltered raw data were 5917 stations with observed DIN:DOC/TOC:SRP (OPO4/DIP) data available.


## 2. Now look at filtered/aggregated raw data: number of stations with available DIN:DOC/TOC:SRP data should be increased by aggregation: 

In [27]:
filtered_raw_data = filtered_raw_data.reset_index()
filtered_raw_data['dataset']=filtered_raw_data['dataset'].astype(str)
filtered_raw_data['site_id'] = filtered_raw_data['site_id'].astype(str)
filtered_raw_data = filtered_raw_data.set_index(['dataset', 'site_id'])
filtered_raw_data['station_ID'] = filtered_raw_data.index
filtered_raw_data['station_ID'] = filtered_raw_data['station_ID'].astype(str)


In [28]:
filtered_raw_data.sort_values(by =  ['station_ID', 'obs_date'])
filtered_available = filtered_raw_data[(filtered_raw_data['NO3N'].notna()|filtered_raw_data['NO2N_NO3N'].notna())& (filtered_raw_data['NH4N'].notna())&(filtered_raw_data['TOC'].notna()|filtered_raw_data['DOC'].notna())&(filtered_raw_data['DIP'].notna()|filtered_raw_data['OPO4'].notna())]['station_ID'].unique()

print(f'In filtered and aggregated raw data are {len(filtered_available)} stations with observed DIN:DOC/TOC:SRP (OPO4/DIP) data available. The number of available stations increased by aggregation by {len(filtered_available)-len(raw_available)} stations.')
#filtered_raw_data[(filtered_raw_data['NO3N'].notna()|filtered_raw_data['NO2N_NO3N'].notna())& (filtered_raw_data['NH4N'].notna())&(filtered_raw_data['TOC'].notna()|filtered_raw_data['DOC'].notna())&(filtered_raw_data['DIP'].notna()|filtered_raw_data['OPO4'].notna())]

In filtered and aggregated raw data are 6050 stations with observed DIN:DOC/TOC:SRP (OPO4/DIP) data available. The number of available stations increased by aggregation by 133 stations.


## Now check where these additional stations are located: 

In [29]:
additional_stations =np.setdiff1d(filtered_available, raw_available)
print(len(np.setdiff1d(filtered_available, raw_available)))


133


In [30]:
# read in stations: 
stations = gpd.read_file('../../output_data/merged_datasets/stations.shp')
stations = stations.set_index(['dataset','site_id']).sort_index()



In [29]:
stations['ids'] = stations.index
stations['ids'] = stations['ids'].astype(str)
print(len(stations[stations['ids'].isin(additional_stations)]))
countries = stations[stations['ids'].isin(additional_stations)]['country'].unique().tolist()
print(f'Additional stations are loacated in following countries: {countries}.')

133
Additional stations are loacated in following countries: ['Brazil', 'Argentina', 'SWEDEN', 'Germany', 'Kenia', 'Denmark'].
